# 1.6 Callback 基礎

這份 Notebook 示範 Keras 常用 callback：EarlyStopping、ModelCheckpoint 與 ReduceLROnPlateau。


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

tf.keras.utils.set_random_seed(42)


## 1. 載入二元分類資料


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.values.astype('float32')
y = data.target.values.astype('float32')

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print('x_train shape:', x_train.shape)
print('positive ratio:', y_train.mean().round(3))


## 2. 建立模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)


## 3. 設定 Callback


In [ ]:
checkpoint_path = 'best_callback_model.keras'

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0.001,
        patience=8,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-5
    )
]


## 4. 執行訓練


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=120,
    batch_size=32,
    callbacks=callbacks,
    verbose=0
)

print('實際訓練 epochs:', len(history.history['loss']))
print('最佳 val_loss:', np.min(history.history['val_loss']).round(4))


## 5. 評估目前模型與最佳 checkpoint


In [ ]:
train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

best_model = tf.keras.models.load_model(checkpoint_path)
best_test_result = best_model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, test_result, best_test_result], index=['current_train', 'current_test', 'checkpoint_test'])


In [ ]:
y_prob = best_model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, digits=4))


## 6. 觀察訓練曲線


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Callbacks Training Curve')
plt.show()


## 7. 小結

Callback 可以讓訓練流程自動停在較好的時間點、保存最佳模型，並在訓練停滯時降低 learning rate。這些都是正式專案中很常用的訓練控制技巧。
